# Projet : Solveur hybride Newton + Dichotomie

Ce projet compare la méthode de Newton et la dichotomie pour trouver des racines
de fonctions. On implémente un solveur hybride qui détecte quand Newton diverge
ou cycle, et bascule automatiquement sur la dichotomie.


## 1. Les deux méthodes de base


In [ ]:
def dichotomie(f, a, b, tolerance=1e-10):
    if f(a) * f(b) > 0:
        raise ValueError("f(a) et f(b) doivent etre de signes opposes")
    iterations = 0
    while b - a > tolerance:
        m = (a + b) / 2
        if f(a) * f(m) <= 0:
            b = m
        else:
            a = m
        iterations += 1
    return (a + b) / 2, iterations

def newton(f, df, depart, tolerance=1e-10, max_iter=100):
    x = depart
    for i in range(max_iter):
        fx = f(x)
        dfx = df(x)
        if dfx == 0:
            return None, i
        x_new = x - fx / dfx
        if abs(x_new - x) < tolerance:
            return x_new, i + 1
        x = x_new
    return None, max_iter


## 2. Test sur des fonctions simples


In [ ]:
import math

fonctions = [
    ("x^2 - 2",    lambda x: x**2 - 2,       lambda x: 2*x,      1.0, 2.0, 1.0),
    ("x^3 - 2x-5", lambda x: x**3 - 2*x - 5, lambda x: 3*x**2-2, 2.0, 3.0, 2.0),
    ("sin(x)",     math.sin,                  math.cos,           3.0, 4.0, 3.5),
]

print(f"{'Fonction':<15} {'Newton iter':>12} {'Dicho iter':>12} {'Racine':>15}")
print("-" * 55)
for nom, f, df, a, b, dep in fonctions:
    r_n, it_n = newton(f, df, dep)
    r_d, it_d = dichotomie(f, a, b)
    print(f"{nom:<15} {it_n:>12} {it_d:>12} {r_d:>15.10f}")


## 3. Cas où Newton échoue : cycle entre 0 et 1

Le polynôme $x^3 - 2x + 2$ n'a pas de racine réelle sur $[0, 1]$.
Depuis $x_0 = 0$, Newton alterne entre 0 et 1 sans jamais converger.


In [ ]:
f_cycle = lambda x: x**3 - 2*x + 2
df_cycle = lambda x: 3*x**2 - 2

x = 0.0
print("Trajet de Newton depuis 0 :")
for i in range(12):
    fx = f_cycle(x)
    dfx = df_cycle(x)
    print(f"  iter {i}: x = {x:.6f}")
    if dfx == 0:
        break
    x = x - fx / dfx


## 4. Solveur hybride

Le solveur hybride lance Newton. Si au bout de `patience` itérations la
valeur absolue n'a pas diminué, ou si on détecte un cycle, il bascule sur la
dichotomie sur l'intervalle donné.


In [ ]:
def solveur_hybride(f, df, a, b, tolerance=1e-10, patience=10):
    x = (a + b) / 2
    historique = []
    for i in range(patience):
        fx = f(x)
        dfx = df(x)
        if dfx == 0:
            break
        x_new = x - fx / dfx
        if abs(f(x_new)) < abs(fx) and a <= x_new <= b:
            historique.append(x_new)
            if len(historique) >= 3 and abs(historique[-1] - historique[-3]) < tolerance:
                print(f"  Cycle detecte a l'iter {i}, bascule sur dichotomie")
                r, it = dichotomie(f, a, b, tolerance)
                return r, i + it, "hybride"
            if abs(x_new - x) < tolerance:
                return x_new, i + 1, "newton"
            x = x_new
        else:
            print(f"  Newton diverge a l'iter {i}, bascule sur dichotomie")
            r, it = dichotomie(f, a, b, tolerance)
            return r, i + it, "hybride"
    print(f"  patience epuisee, bascule sur dichotomie")
    r, it = dichotomie(f, a, b, tolerance)
    return r, patience + it, "hybride"

print("x^2 - 2 sur [1, 2]:")
r, it, methode = solveur_hybride(lambda x: x**2-2, lambda x: 2*x, 1.0, 2.0)
print(f"  racine={r:.10f}, {it} iters, methode={methode}")

print("x^3 - 2x + 2 sur [-2, 0] (racine reelle):")
r, it, methode = solveur_hybride(f_cycle, df_cycle, -2.0, 0.0)
print(f"  racine={r:.10f}, {it} iters, methode={methode}")


## 5. Comparaison du nombre d'itérations

On compare Newton, dichotomie et le solveur hybride sur plusieurs fonctions
et plusieurs points de départ.


In [ ]:
cas = [
    ("x^2-2",      lambda x: x**2-2,        lambda x: 2*x,       1.0, 2.0),
    ("x^3-2x-5",   lambda x: x**3-2*x-5,    lambda x: 3*x**2-2,  2.0, 3.0),
    ("x^3-2x+2",   lambda x: x**3-2*x+2,    lambda x: 3*x**2-2, -2.0, 0.0),
]

print(f"{'Fonction':<14} {'Newton':>8} {'Dicho':>8} {'Hybride':>9}")
print("-" * 42)
for nom, f, df, a, b in cas:
    r_n, it_n = newton(f, df, (a+b)/2)
    r_d, it_d = dichotomie(f, a, b)
    r_h, it_h, _ = solveur_hybride(f, df, a, b)
    it_n_str = str(it_n) if r_n is not None else "echec"
    print(f"{nom:<14} {it_n_str:>8} {it_d:>8} {it_h:>9}")


## 6. Conclusion

Newton est très rapide quand il converge (moins de 10 itérations), mais il peut
cycler ou diverger selon le point de départ. La dichotomie est robuste mais plus
lente (environ 33 itérations pour une précision de $10^{-10}$). Le solveur
hybride tente Newton d'abord et bascule automatiquement sur la dichotomie si
nécessaire : il combine robustesse et rapidité.
